# Sliding-Window Feature Extraction — Escalation-Stage Pipeline

Foundational step for the reframed architecture (clip-level classification
-> per-window escalation-stage prediction), per Geetanjali's 2026-11-09
review: the current fusion architecture isn't novel (frozen-backbone
fusion is already published -- VadCLIP, AnomalyCLIP, CLIP-TSA all do this
on XD-Violence), and she separately asked for a live color-coded
(green/amber/red) risk indicator. Both point at the same fix: stop
classifying whole clips, start modeling risk *as it evolves over time*.

**What this notebook produces:** for every riot and normal clip (both
official train and test split -- the MIL trainer needs the full train
set; the test split additionally has real frame-level onset annotations
for evaluation), a sequence of per-window embeddings instead of one
vector per clip. DINOv2 + OpenCLIP only -- the two branches confirmed to
carry real signal (`docs/findings.md`); LK/RF-DETR/VideoMAE are not
re-extracted into this pipeline given the ablation results.

**Output:** one `.npz` per clip
(`My Drive/CVPHG/sliding_window_features/<video_id>.npz`) holding window
start/end frame indices plus per-window DINOv2 and OpenCLIP embeddings,
and one manifest CSV indexing them all. Resumable -- skips any clip whose
`.npz` already exists.

**Not yet built, next steps after this:** the MIL weakly-supervised
trainer (video-level labels only, since per-window ground truth only
exists for the 101 annotated test-split riot clips), the ordinal 3-tier
sequence model, and the lead-time/MOR-style evaluation against those real
annotations.

## Step 0 — Setup

In [ ]:
!pip install opencv-python-headless open_clip_torch -q

In [ ]:
import csv
import os

import cv2
import numpy as np
import pandas as pd
import torch
from google.colab import drive

drive.mount('/content/drive')

RIOT_DIR = '/content/drive/MyDrive/CVPHG/xdviolence_riot'
NORMAL_DIR = '/content/drive/MyDrive/CVPHG/xdviolence_normal'
OUT_DIR = '/content/drive/MyDrive/CVPHG/sliding_window_features'
os.makedirs(OUT_DIR, exist_ok=True)

# Window sizing in seconds, not frames -- clips in this dataset vary in fps
# (movie vs. YouTube sources), so a frame-count window would cover a
# different real duration per clip. A time-based window keeps "2 seconds
# of context" meaning the same thing across the whole dataset.
WINDOW_SECONDS = 2.0
STRIDE_SECONDS = 1.0  # 50% overlap between consecutive windows

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cpu':
    print('WARNING: no GPU detected. Runtime -> Change runtime type -> T4 GPU recommended '
          '-- this extracts many more windows per clip than the old sparse-frame pipeline.')

In [ ]:
riot_manifest = pd.read_csv(os.path.join(RIOT_DIR, 'manifest_xdviolence_riot.csv'))
riot_manifest['clip_dir'] = RIOT_DIR
normal_manifest = pd.read_csv(os.path.join(NORMAL_DIR, 'manifest_xdviolence_normal.csv'))
normal_manifest['clip_dir'] = NORMAL_DIR

all_clips = pd.concat([riot_manifest, normal_manifest], ignore_index=True)
print(f"{len(riot_manifest)} riot + {len(normal_manifest)} normal = {len(all_clips)} clips total")
print(all_clips.groupby(['label', 'split']).size())

## Step 1 — Load DINOv2 and OpenCLIP (frozen, no fine-tuning)

In [ ]:
dinov2_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
dinov2_model.eval().to(DEVICE)

DINOV2_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
DINOV2_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
DINOV2_DIM = 768


def dinov2_embed_frame(frame_rgb: np.ndarray) -> np.ndarray:
    img = cv2.resize(frame_rgb, (224, 224), interpolation=cv2.INTER_AREA)
    tensor = torch.from_numpy(img).permute(2, 0, 1).float().unsqueeze(0) / 255.0
    tensor = (tensor - DINOV2_MEAN) / DINOV2_STD
    tensor = tensor.to(DEVICE)
    with torch.no_grad():
        feat = dinov2_model(tensor)
    return feat.squeeze(0).cpu().numpy()

In [ ]:
import open_clip
from PIL import Image

openclip_model, _, openclip_preprocess = open_clip.create_model_and_transforms('ViT-L-14', pretrained='openai')
openclip_model.eval().to(DEVICE)
OPENCLIP_DIM = 768


def openclip_embed_frame(frame_rgb: np.ndarray) -> np.ndarray:
    img = Image.fromarray(frame_rgb)
    tensor = openclip_preprocess(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        feat = openclip_model.encode_image(tensor)
        feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.squeeze(0).cpu().numpy()

## Step 2 — Window the clip and embed the center frame of each window

One representative frame per window (the center frame) is embedded, not
every frame -- this keeps extraction cost comparable to the earlier
sparse-sampling pipeline while still producing a real sequence across the
clip, instead of one pooled vector for the whole thing.

In [ ]:
def compute_windows(video_path):
    """Returns a list of (start_frame, end_frame, center_frame) for
    windows of WINDOW_SECONDS, stepped every STRIDE_SECONDS, covering the
    full clip duration."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    if fps <= 0 or total_frames <= 0:
        return [], fps, total_frames

    window_frames = max(1, int(round(WINDOW_SECONDS * fps)))
    stride_frames = max(1, int(round(STRIDE_SECONDS * fps)))

    windows = []
    start = 0
    while start < total_frames:
        end = min(start + window_frames, total_frames)
        center = (start + end) // 2
        windows.append((start, end, center))
        if end >= total_frames:
            break
        start += stride_frames
    return windows, fps, total_frames


def read_frame_at(video_path, frame_idx):
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame_bgr = cap.read()
    cap.release()
    if not ok:
        return None
    return cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

## Step 3 — Per-clip extraction loop (resumable)

In [ ]:
manifest_rows = []
already_done = {f.replace('.npz', '') for f in os.listdir(OUT_DIR) if f.endswith('.npz')}
print(f"Resuming: {len(already_done)} clips already have window features.")

for i, row in enumerate(all_clips.to_dict('records'), 1):
    video_id = row['id']
    out_path = os.path.join(OUT_DIR, f"{video_id}.npz")

    if video_id in already_done:
        manifest_rows.append({'video_id': video_id, 'label': row['label'], 'split': row['split']})
    else:
        video_path = os.path.join(row['clip_dir'], video_id)
        try:
            windows, fps, total_frames = compute_windows(video_path)
            if not windows:
                print(f"  SKIPPED {video_id}: unreadable or zero-length")
                continue

            dinov2_feats = np.zeros((len(windows), DINOV2_DIM), dtype=np.float32)
            openclip_feats = np.zeros((len(windows), OPENCLIP_DIM), dtype=np.float32)
            starts = np.zeros(len(windows), dtype=np.int32)
            ends = np.zeros(len(windows), dtype=np.int32)

            for w, (start, end, center) in enumerate(windows):
                frame = read_frame_at(video_path, center)
                if frame is None:
                    continue
                dinov2_feats[w] = dinov2_embed_frame(frame)
                openclip_feats[w] = openclip_embed_frame(frame)
                starts[w] = start
                ends[w] = end

            np.savez_compressed(
                out_path,
                window_start=starts, window_end=ends,
                dinov2=dinov2_feats, openclip=openclip_feats,
                fps=fps, total_frames=total_frames,
            )
            manifest_rows.append({'video_id': video_id, 'label': row['label'], 'split': row['split']})
        except Exception as e:
            print(f"  FAILED {video_id}: {e}")
            continue

    if i % 25 == 0 or i == len(all_clips):
        print(f"  {i}/{len(all_clips)} clips processed")

manifest_df = pd.DataFrame(manifest_rows)
manifest_path = os.path.join(OUT_DIR, 'manifest_sliding_window.csv')
manifest_df.to_csv(manifest_path, index=False)
print(f"\nDone. {len(manifest_df)}/{len(all_clips)} clips have window features.")
print(f"Manifest -> {manifest_path}")

## Sanity check — window counts look right?

A handful of clips at a few different durations, to confirm the window
count scales the way it should before trusting this for the full run.

In [ ]:
sample_ids = manifest_df['video_id'].sample(min(5, len(manifest_df)), random_state=42).tolist()
for vid in sample_ids:
    data = np.load(os.path.join(OUT_DIR, f"{vid}.npz"))
    duration_s = data['total_frames'] / data['fps'] if data['fps'] > 0 else 0
    print(f"{vid}: {duration_s:.1f}s clip -> {len(data['window_start'])} windows "
          f"(fps={float(data['fps']):.1f})")

## Next steps

- **MIL weakly-supervised trainer** (not yet built): the official train
  split has no frame-level labels -- only a whole-clip "contains a riot
  somewhere" tag -- so training needs a multiple-instance-learning setup
  (bag = all windows of one clip, video-level label only, a ranking-style
  loss encouraging the highest-scoring window in a riot clip to outscore
  the highest-scoring window in a normal clip).
- **Ordinal 3-tier sequence model** (not yet built): a small GRU/attention
  model over each clip's window sequence, predicting Normal/Escalating/
  Critical per window.
- **Lead-time evaluation** (not yet built): using the 101 officially
  annotated test-split riot clips' real onset frames
  (`annotations.txt`, see `docs/findings.md` 2026-09-18 entry) --
  adapting Minimum Observation Ratio (Furnari & Farinella) and/or the
  early-time-series-classification harmonic-mean(accuracy, 1-earliness)
  framework, per `docs/roadmap.md`'s 2026-11-09 research findings.
- **Confound check, do this before trusting any early-detection result:**
  verify the model isn't picking up source-level artifacts (title cards,
  watermarks, compression differences) rather than genuine escalating
  content -- known issue in this exact literature (see `docs/roadmap.md`,
  the "Pre-Event Separability" paper citation).